In [18]:
import pandas as pd
import numpy as np
import yfinance as yf
from matplotlib import pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from plotly.subplots import make_subplots
from plotly import graph_objects as go 

In [19]:
def plot_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows = 2, cols = 1, shared_xaxes= True, subplot_titles=('Historical Shared Pricce ($)',
    'Historical Revenue ($)'), vertical_spacing=.5)
    fig.add_trace(go.Scatter(x = pd.to_datetime(stock_data['Date'],),
                             y = stock_data['Close'].astype('float'),name = 'Shared Price'), row = 1, col =1)
    fig.add_trace(go.Scatter(x = pd.to_datetime(revenue_data['Date'],),
                            y = revenue_data['Revenue'].astype('float'), name = 'Revenue'), row=2,col=1)
    fig.update_xaxes(title_text ='Date', row = 1,col=1)
    fig.update_xaxes(title_text = 'Date', row = 2, col =1)
    fig.update_yaxes(title_text ='Price ($)', row = 1, col = 1)
    fig.update_yaxes(title_text = 'Revenue ($ Millions)', row = 2, col = 1)
    fig.update_layout(showlegend = False, height = 1000, title = stock, xaxis_rangeslider_visible = True)
    fig.show()

In [20]:
def candle_stick(stock_data, stock):
    fig = go.Figure(data = [go.Candlestick(x = stock_data['Date'], open = stock_data['Open'],
                    high = stock_data['High'], low = stock_data['Low'], close = stock_data['Close'])],
                   )
    fig.update_xaxes(rangeslider_visible = True, rangeselector = dict(
        buttons = list([
            dict(count = 1, label = '1m', step = 'month', stepmode = 'backward'),
            dict(count = 6, label = '6m', step = 'month', stepmode = 'backward'),
            dict(count = 1, label = 'YTD', step = 'year', stepmode = 'todate'),
            dict(count = 1, label = '1y', step = 'year', stepmode = 'backward'),
            dict(step = 'all'),
        ])
    ))
    fig.update_layout(title =f'{stock} Stock Price ($)')
    fig.show()

In [21]:
tesla_data = yf.Ticker('TSLA')
tesla_data = tesla_data.history(period = 'max')
tesla_data.reset_index(inplace = True)
tesla_data

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0
...,...,...,...,...,...,...,...,...
3641,2024-12-16 00:00:00-05:00,441.089996,463.190002,436.149994,463.019989,114083800,0.0,0.0
3642,2024-12-17 00:00:00-05:00,475.899994,483.989990,457.510010,479.859985,131223000,0.0,0.0
3643,2024-12-18 00:00:00-05:00,466.500000,488.540009,427.010010,440.130005,149340800,0.0,0.0
3644,2024-12-19 00:00:00-05:00,451.880005,456.359985,420.019989,436.170013,118566100,0.0,0.0


In [22]:
url = ('https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue')
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'}
html = requests.get(url, headers = headers)
soup = BeautifulSoup(html.text, 'html.parser')

In [23]:
date = []
revenue = []
tables = soup.find_all('table')
table_index = 0
for index, table in enumerate(tables):
    if 'Tesla Quarterly Revenue' in str(table):
        table_index = index
for row in tables[table_index].tbody.find_all('tr'):
    col = row.find_all('td')
    #print(col)
    if (col!=[]):
        date.append(col[0].text)
        revenue.append(col[1].text.replace('$','').replace(',',''))
tesla_revenue = pd.DataFrame({'Date': date, 'Revenue': revenue})
tesla_revenue
        

,Date,Revenue
0,2024-09-30,25182
1,2024-06-30,25500
2,2024-03-31,21301
3,2023-12-31,25167
4,2023-09-30,23350
...,...,...
57,2010-06-30,28
58,2010-03-31,21
59,2009-12-31,
60,2009-09-30,46


In [24]:
tesla_revenue = tesla_revenue[tesla_revenue['Revenue']!='']

In [25]:
tesla_revenue.info()

<class 'pandas.core.frame.DataFrame'>
Index: 61 entries, 0 to 61
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     61 non-null     object
 1   Revenue  61 non-null     object
dtypes: object(2)
memory usage: 1.4+ KB


In [26]:
tesla_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3646 entries, 0 to 3645
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype                           
---  ------        --------------  -----                           
 0   Date          3646 non-null   datetime64[ns, America/New_York]
 1   Open          3646 non-null   float64                         
 2   High          3646 non-null   float64                         
 3   Low           3646 non-null   float64                         
 4   Close         3646 non-null   float64                         
 5   Volume        3646 non-null   int64                           
 6   Dividends     3646 non-null   float64                         
 7   Stock Splits  3646 non-null   float64                         
dtypes: datetime64[ns, America/New_York](1), float64(6), int64(1)
memory usage: 228.0 KB


In [27]:
plot_graph(tesla_data, tesla_revenue, 'Tesla Historical Shared Price & Revenue')

In [28]:
candle_stick(tesla_data,'Tesla')

In [29]:
gamestop = yf.Ticker('GME')
gme_data = gamestop.history(period = 'max')
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683251,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666417,1.666417,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662209,1.603296,1.662209,6892800,0.0,0.0


In [30]:
url = 'https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue'
html = requests.get(url, headers = headers)
soup = BeautifulSoup(html.text,'html.parser')

In [31]:
date, revenue = [], []
tables = soup.find_all('table')
table_index = 0
for index, table in enumerate(tables):
    if "GameStop Quarterly Revenue" in str(table):
        table_index = index
        
for row in tables[table_index].tbody.find_all('tr'):
    col = row.findAll('td')
    if col != []:
        date.append(col[0].text)
        revenue.append(col[1].text.replace('$','').replace(',',''))
gme_revenue = pd.DataFrame({'Date': date, 'Revenue': revenue})
gme_revenue

,Date,Revenue
0,2024-07-31,798
1,2024-04-30,882
2,2024-01-31,1794
3,2023-10-31,1078
4,2023-07-31,1164
...,...,...
58,2010-01-31,3524
59,2009-10-31,1835
60,2009-07-31,1739
61,2009-04-30,1981


In [32]:
gme_revenue = gme_revenue[gme_revenue['Revenue'] != '']
gme_revenue

,Date,Revenue
0,2024-07-31,798
1,2024-04-30,882
2,2024-01-31,1794
3,2023-10-31,1078
4,2023-07-31,1164
...,...,...
58,2010-01-31,3524
59,2009-10-31,1835
60,2009-07-31,1739
61,2009-04-30,1981


In [33]:
plot_graph(gme_data,gme_revenue,'GME Historical Shared Price & Revenue')

In [34]:
candle_stick(gme_data,'GME')